# ETL BANCO SHIELD
Pipeline de tratamento, padronização e validação dos dados analíticos utilizados para construção do ambiente BI.

In [450]:
import pandas as pd
import numpy as np

## FATO_CONTRATOS
Tabela fato responsável pela consolidação dos eventos financeiros, operacionais e indicadores de risco da carteira.
- contract_id (PK lógica, texto): Identificador do contrato/registro (espera-se unicidade).
- ano_mes (inteiro AAAAMM): Mês de referência do evento.
- bank (texto): Instituição (Banco Shield ou concorrente Hidra).
- product_id (FK -> dim_produto.product_id): Produto contratado.
- location_id (FK -> dim_localidade.location_id): Localidade de originação.
- units (inteiro): Quantidade de contratos no registro (neste dataset, 1 por linha).
- financed_amount (decimal): Valor financiado/contratado no mês (R$). Para produtos sem contratação financeira (ex.: Conta), pode ser 0.
- outstanding_balance (decimal): Saldo/exposição em aberto (R$).
- dpd (inteiro): Days Past Due (dias em atraso). 0 = adimplente.
- delinquent_amount_30p (decimal): Valor em atraso (R$) para contratos 30+ DPD (0 quando não aplicável).
- risk_score (decimal 0-1): Score de risco sintético (maior = mais risco).

### Criando dataframe.
Inicialização da camada de ingestão e estruturação dos dados transacionais.

In [451]:
df_fato_contratos = pd.read_csv(f'C:/Users/vitoo/Banco_Shield/fato_contratos.csv')

In [452]:
df_fato_contratos.head(5)

,contract_id,ano_mes,bank,product_id,location_id,units,financed_amount,outstanding_balance,dpd,delinquent_amount_30p,risk_score
0,C202501-BA-000001,202501,Banco Shield,1013.0,509,1,24668.11,19184.31,0,0.0,0.0842
1,C202501-BA-000002,202501,Banco Shield,1008.0,508,1,15908.45,16095.09,0,0.0,0.1144
2,C202501-BA-000003,202501,Banco Shield,1009.0,508,1,0.00,0.00,0,0.0,0.0864
3,C202501-BA-000004,202501,Banco Shield,1003.0,501,1,13372.75,9564.60,0,0.0,0.0886
4,C202501-BA-000005,202501,Banco Shield,1009.0,515,1,0.00,0.00,0,0.0,0.0756


In [453]:
df_fato_contratos.info()
df_fato_contratos.describe()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   contract_id            6000 non-null   str    
 1   ano_mes                6000 non-null   int64  
 2   bank                   6000 non-null   str    
 3   product_id             5985 non-null   float64
 4   location_id            6000 non-null   int64  
 5   units                  6000 non-null   int64  
 6   financed_amount        5984 non-null   float64
 7   outstanding_balance    6000 non-null   float64
 8   dpd                    6000 non-null   int64  
 9   delinquent_amount_30p  6000 non-null   float64
 10  risk_score             6000 non-null   float64
dtypes: float64(5), int64(4), str(2)
memory usage: 515.8 KB


,ano_mes,product_id,location_id,units,financed_amount,outstanding_balance,dpd,delinquent_amount_30p,risk_score
count,6000.000000,5985.000000,6000.000000,6000.0,5984.000000,6000.000000,6000.000000,6000.000000,6000.000000
mean,202506.523167,1010.492063,508.839000,1.0,14258.570259,11850.168082,0.951667,66.714408,0.114559
std,3.466718,5.754850,20.489332,0.0,17021.703950,14145.837593,5.055386,1068.990333,0.043073
min,202501.000000,1001.000000,501.000000,1.0,-74180.380000,0.000000,0.000000,0.000000,0.010000
25%,202504.000000,1005.000000,504.000000,1.0,708.870000,584.847500,0.000000,0.000000,0.083875
50%,202507.000000,1011.000000,508.000000,1.0,6725.740000,5649.120000,0.000000,0.000000,0.112450
75%,202510.000000,1015.000000,512.000000,1.0,23861.492500,19009.215000,0.000000,0.000000,0.144000
max,202513.000000,1020.000000,999.000000,1.0,85424.120000,77274.120000,90.000000,34143.700000,0.283000


### Data quality

#### Remoção de nulos
Tratamento de registros incompletos para garantir maior confiabilidade analítica.

In [454]:
df_fato_contratos_dropnulos = df_fato_contratos.dropna()

In [455]:
df_fato_contratos.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   contract_id            6000 non-null   str    
 1   ano_mes                6000 non-null   int64  
 2   bank                   6000 non-null   str    
 3   product_id             5985 non-null   float64
 4   location_id            6000 non-null   int64  
 5   units                  6000 non-null   int64  
 6   financed_amount        5984 non-null   float64
 7   outstanding_balance    6000 non-null   float64
 8   dpd                    6000 non-null   int64  
 9   delinquent_amount_30p  6000 non-null   float64
 10  risk_score             6000 non-null   float64
dtypes: float64(5), int64(4), str(2)
memory usage: 515.8 KB


#### Conversão de tipos
Padronização dos tipos de dados conforme especificação técnica do modelo analítico.

In [456]:
df_fato_contratos_dropnulos['contract_id'] = df_fato_contratos_dropnulos['contract_id'].astype(str)
df_fato_contratos_dropnulos['ano_mes'] = df_fato_contratos_dropnulos['ano_mes'].astype(int)
df_fato_contratos_dropnulos['bank'] = df_fato_contratos_dropnulos['bank'].astype(str)
df_fato_contratos_dropnulos['product_id'] = df_fato_contratos_dropnulos['product_id'].astype(int)
df_fato_contratos_dropnulos['location_id'] = df_fato_contratos_dropnulos['location_id'].astype(int)

In [457]:
df_fato_contratos_dropnulos['units'] = df_fato_contratos_dropnulos['units'].astype(int)
df_fato_contratos_dropnulos['financed_amount'] = df_fato_contratos_dropnulos['financed_amount'].astype(float)
df_fato_contratos_dropnulos['outstanding_balance'] = df_fato_contratos_dropnulos['outstanding_balance'].astype(float)
df_fato_contratos_dropnulos['dpd'] = df_fato_contratos_dropnulos['dpd'].astype(int)
df_fato_contratos_dropnulos['delinquent_amount_30p'] = df_fato_contratos_dropnulos['delinquent_amount_30p'].astype(float)
df_fato_contratos_dropnulos['risk_score'] = df_fato_contratos_dropnulos['risk_score'].astype(float)

#### Validação do risk_score
Validação de integridade da chave primária lógica dos contratos.

In [458]:
df_fato_contratos_dropnulos['risk_score'] = (
    df_fato_contratos_dropnulos['risk_score'].clip(0, 1)
)

In [459]:
print(df_fato_contratos['risk_score'].head(20))

0     0.0842
1     0.1144
2     0.0864
3     0.0886
4     0.0756
5     0.0511
6     0.1321
7     0.1083
8     0.0936
9     0.1016
10    0.0811
11    0.0488
12    0.1232
13    0.0713
14    0.1205
15    0.1178
16    0.0966
17    0.0873
18    0.0530
19    0.0861
Name: risk_score, dtype: float64


#### Verificação de unicidade
Validação de integridade da chave primária lógica dos contratos.

In [460]:
duplicados = df_fato_contratos_dropnulos.duplicated(subset=['contract_id']).sum()
if duplicados > 0:
    print(f"Atenção: Foram encontrados {duplicados} registros duplicados de contract_id. Removendo duplicidades...")
    df_fato_contratos_dropnulos = df_fato_contratos_dropnulos.drop_duplicates(subset=['contract_id'], keep='first')

Atenção: Foram encontrados 16 registros duplicados de contract_id. Removendo duplicidades...


In [461]:
print(df_fato_contratos_dropnulos.dtypes)

contract_id                  str
ano_mes                    int64
bank                         str
product_id                 int64
location_id                int64
units                      int64
financed_amount          float64
outstanding_balance      float64
dpd                        int64
delinquent_amount_30p    float64
risk_score               float64
dtype: object


### Regras de negócio esperadas

Aplicação das regras funcionais e restrições operacionais definidas para o domínio de negócio.
- Integridade referencial: product_id e location_id devem existir nas dimensões.
- Domínios válidos:
- bank deve ser "Banco Shield" ou "Hidra".
- ano_mes deve estar no intervalo 202501 a 202512.
- Regras de valores:
- financed_amount e outstanding_balance não devem ser negativos.
- delinquent_amount_30p deve ser 0 quando dpd < 30.
- risk_score deve estar entre 0 e 1.
- Unicidade: contract_id deve ser único.

#### Correção contract_id x bank
Correção de inconsistências cadastrais entre identificadores de contrato e instituição financeira associada.

In [462]:
def corrigir_nome_banco(row):
    prefixo = str(row['contract_id'])[:2].lower()
    if prefixo == 'ba':
        return 'Banco Shield'
    elif prefixo == 'hi':
        return 'Hidra'
    else:
        return row['bank']

In [463]:
df_fato_contratos_dropnulos['bank'] = df_fato_contratos_dropnulos.apply(corrigir_nome_banco, axis=1)

##### Integridade referencial

Validação de consistência entre tabelas fato e dimensões do modelo analítico.

In [464]:
df_dim_produto = pd.read_csv('dim_produto.csv') 
df_dim_localidade = pd.read_csv('dim_localidade.csv')

In [465]:
df_fato_contratos_dropnulos = df_fato_contratos_dropnulos[df_fato_contratos_dropnulos['product_id'].isin(df_dim_produto['product_id'])]
df_fato_contratos_dropnulos = df_fato_contratos_dropnulos[df_fato_contratos_dropnulos['location_id'].isin(df_dim_localidade['location_id'])]

##### Domínios válidos

Padronização e validação dos valores permitidos nas variáveis categóricas.

In [466]:
bancos_validos = ["Banco Shield", "Hidra"]
df_fato_contratos_dropnulos = df_fato_contratos_dropnulos[df_fato_contratos_dropnulos['bank'].isin(bancos_validos)]

Garantindo que ano_mes devem estar no intervalo 202501 a 202512.

In [467]:
df_fato_contratos_dropnulos = df_fato_contratos_dropnulos[df_fato_contratos_dropnulos['ano_mes'].between(202501, 202512)]

##### Regras de valores

Validação de consistência financeira e prevenção de valores inválidos na carteira.

In [468]:
df_fato_contratos_dropnulos['financed_amount'] = df_fato_contratos_dropnulos['financed_amount'].clip(lower=0)
df_fato_contratos_dropnulos['outstanding_balance'] = df_fato_contratos_dropnulos['outstanding_balance'].clip(lower=0)

Aplicação da lógica de inadimplência para contratos abaixo de 30 dias de atraso.

In [469]:
df_fato_contratos_dropnulos.loc[df_fato_contratos_dropnulos['dpd'] < 30, 'delinquent_amount_30p'] = 0.0

❗Regras de negócio como a unicidade do contract_id e o intervalo do risk_score não foram aplicadas nesta etapa, pois já foram tratadas anteriormente na fase de Data quality.❗

In [470]:
print(df_fato_contratos_dropnulos.dtypes)
print(f"Total de registros após tratamento: {len(df_fato_contratos_dropnulos)}")

contract_id                  str
ano_mes                    int64
bank                         str
product_id                 int64
location_id                int64
units                      int64
financed_amount          float64
outstanding_balance      float64
dpd                        int64
delinquent_amount_30p    float64
risk_score               float64
dtype: object
Total de registros após tratamento: 5902


## DIM_PRODUTO
Dimensão responsável pela categorização e detalhamento dos produtos financeiros.
- product_id (PK, inteiro): Identificador único do produto.
- product_name (texto): Nome do produto no universo Marvel.
- category (texto): Categoria do produto (ex.: Financiamento, Cartões, Seguro etc.).
- tenor_months (inteiro): Prazo típico do produto, em meses.
- base_rate_apr (decimal): Taxa base anual (APR) usada como referência.

### Criando Dataframe.

In [471]:
df_dim_produto = pd.read_csv(f'C:/Users/vitoo/Banco_Shield/dim_produto.csv')

In [472]:
print(df_dim_produto)

    product_id                    product_name       category  tenor_months  \
0         1001                Cartão Vibranium          Conta            60   
1         1002           Financiamento Quinjet         Seguro            24   
2         1003                 Consórcio Stark     Empréstimo            48   
3         1004                   Seguro Asgard          Conta            24   
4         1005          Empréstimo Arc Reactor      Consórcio            48   
5         1006              Leasing Hulkbuster  Investimentos            60   
6         1007               Crédito Wakandano     Empréstimo            12   
7         1008            Plano Família Barton     Empréstimo            48   
8         1009            Conta Digital SHIELD          Conta            24   
9         1010           Investimento Infinity  Financiamento            60   
10        1011         Refinanciamento Mjölnir      Consórcio            48   
11        1012                 Crédito Pantera      

In [473]:
df_dim_produto.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     20 non-null     int64  
 1   product_name   20 non-null     str    
 2   category       20 non-null     str    
 3   tenor_months   20 non-null     int64  
 4   base_rate_apr  20 non-null     float64
dtypes: float64(1), int64(2), str(2)
memory usage: 932.0 bytes


### Data quality

Tipagem e Saneamento da Dimensão Produto

In [474]:
df_dim_produto['product_id'] = df_dim_produto['product_id'].astype(int)
df_dim_produto['product_name'] = df_dim_produto['product_name'].astype(str).str.strip()
df_dim_produto['category'] = df_dim_produto['category'].astype(str).str.strip()
df_dim_produto['tenor_months'] = df_dim_produto['tenor_months'].astype(int).clip(lower=0)
df_dim_produto['base_rate_apr'] = df_dim_produto['base_rate_apr'].astype(float).clip(lower=0)

Removendo espaços em branco extras e padronizando nomes.

In [475]:
df_dim_produto['product_name'] = df_dim_produto['product_name'].str.strip()
df_dim_produto['category'] = df_dim_produto['category'].str.strip()

Como product_id é a PK, a unicidade é obrigatória.

In [476]:
duplicados_prod = df_dim_produto.duplicated(subset=['product_id']).sum()
if duplicados_prod > 0:
    print(f"Aviso: {duplicados_prod} IDs de produto duplicados encontrados. Removendo...")
    df_dim_produto = df_dim_produto.drop_duplicates(subset=['product_id'], keep='first')

Garantindo que prazos e taxas não sejam negativos.

In [477]:
df_dim_produto['tenor_months'] = df_dim_produto['tenor_months'].clip(lower=0)
df_dim_produto['base_rate_apr'] = df_dim_produto['base_rate_apr'].clip(lower=0)

In [478]:
print("Produto tratado.")
print(df_dim_produto.dtypes)

Produto tratado.
product_id         int64
product_name         str
category             str
tenor_months       int64
base_rate_apr    float64
dtype: object


## DIM_LOCALIDADE
Dimensão responsável pela segmentação geográfica e regional das operações.
- location_id (PK, inteiro): Identificador único da localidade.
- location_name (texto): Nome da localidade (cidades/territórios do universo Marvel e Brasil).
- macro_region (texto): Macro-região (ex.: Brasil, Europa, Galáxia etc.).
- risk_factor_region (decimal): Fator regional que influencia risco (quanto maior, maior propensão a inadimplência).


### Criando Dataframe

In [479]:
df_dim_localidade = pd.read_csv(f'C:/Users/vitoo/Banco_Shield/dim_localidade.csv')

In [480]:
print(df_dim_localidade)

    location_id   location_name   macro_region  risk_factor_region
0           501       Nova York  Norte América               1.034
1           502         Wakanda         África               1.046
2           503     Nova Asgard         Europa               0.883
3           504         Sokovia         Europa               1.236
4           505          Xandar        Galáxia               1.149
5           506        Knowhere        Galáxia               1.223
6           507      Sanctum SP         Brasil               1.203
7           508  Rio de Janeiro         Brasil               1.069
8           509       São Paulo         Brasil               1.215
9           510        Curitiba         Brasil               0.840
10          511        Salvador         Brasil               0.888
11          512  Belo Horizonte         Brasil               0.820
12          513          Recife         Brasil               0.946
13          514    Porto Alegre         Brasil               0

In [481]:
df_dim_localidade.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   location_id         15 non-null     int64  
 1   location_name       15 non-null     str    
 2   macro_region        15 non-null     str    
 3   risk_factor_region  15 non-null     float64
dtypes: float64(1), int64(1), str(2)
memory usage: 612.0 bytes


### Data quality

Definindo chaves como inteiros e descrições como texto.

In [482]:
df_dim_localidade['location_id'] = df_dim_localidade['location_id'].astype(int)
df_dim_localidade['location_name'] = df_dim_localidade['location_name'].astype(str)
df_dim_localidade['macro_region'] = df_dim_localidade['macro_region'].astype(str)

Definindo fator de risco como float.

In [483]:
df_dim_localidade['risk_factor_region'] = df_dim_localidade['risk_factor_region'].astype(float)

Limpeza de espaços em branco para evitar erros de duplicidade visual

In [484]:
df_dim_localidade['location_name'] = df_dim_localidade['location_name'].str.strip()
df_dim_localidade['macro_region'] = df_dim_localidade['macro_region'].str.strip()

Verificação de Unicidade. Essencial para garantir a integridade referencial com a tabela fato.

In [485]:
duplicados_loc = df_dim_localidade.duplicated(subset=['location_id']).sum()
if duplicados_loc > 0:
    print(f"Aviso: {duplicados_loc} IDs de localidade duplicados encontrados. Removendo...")
    df_dim_localidade = df_dim_localidade.drop_duplicates(subset=['location_id'], keep='first')

Como o fator de risco regional influencia a inadimplência, aqui estou garantindo que não seja negativo.

In [486]:
df_dim_localidade['risk_factor_region'] = df_dim_localidade['risk_factor_region'].clip(lower=0)

In [487]:
print("Localidade tratada.")
print(df_dim_localidade.head())

Localidade tratada.
   location_id location_name   macro_region  risk_factor_region
0          501     Nova York  Norte América               1.034
1          502       Wakanda         África               1.046
2          503   Nova Asgard         Europa               0.883
3          504       Sokovia         Europa               1.236
4          505        Xandar        Galáxia               1.149


## Camada final
Camada analítica consolidada para consumo no Power BI e geração de indicadores executivos.

Exportação dos DataFrames tratados para a pasta de dados processados

In [488]:
df_dim_produto.to_csv('dim_produto_processed.csv', index=False, encoding='utf-8')
df_dim_localidade.to_csv('dim_localidade_processed.csv', index=False, encoding='utf-8')
df_fato_contratos_dropnulos.to_csv('fato_contratos_processed.csv', index=False, sep=';', decimal=',', encoding='utf-8')

In [1]:
print("Tratamento dos dados do case finalizados!")

Tratamento dos dados do case finalizados!
